# Data Transformation & Aggregation

In [ ]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
.appName('Olist_data')\
.getOrCreate()

In [ ]:
hdfs_path='/data/olist/'

In [ ]:
customers_df=spark.read.csv(hdfs_path + 'olist_customers_dataset.csv',header=True,inferSchema=True)
orders_df = spark.read.csv(hdfs_path + 'olist_orders_dataset.csv',header=True,inferSchema=True)
order_item_df = spark.read.csv(hdfs_path + 'olist_order_items_dataset.csv',header=True,inferSchema=True)
payments_df = spark.read.csv(hdfs_path + 'olist_order_payments_dataset.csv',header=True,inferSchema=True)
reviews_df = spark.read.csv(hdfs_path + 'olist_order_reviews_dataset.csv',header=True,inferSchema=True)
products_df = spark.read.csv(hdfs_path + 'olist_products_dataset.csv',header=True,inferSchema=True)
sellers_df = spark.read.csv(hdfs_path + 'olist_sellers_dataset.csv',header=True,inferSchema=True)
geolocation_df = spark.read.csv(hdfs_path + 'olist_geolocation_dataset.csv',header=True,inferSchema=True)
category_translation_df = spark.read.csv(hdfs_path + 'product_category_name_translation.csv',header=True,inferSchema=True)


In [ ]:
from pyspark.sql.functions import *

# Optimized Joins For Data Integration

In [ ]:
orders_items_joined_df=orders_df.join(order_item_df,'order_id','inner')
orders_items_products_joined=orders_items_joined_df.join(products_df,'product_id','inner')
orders_items_products_seller_joined=orders_items_products_joined.join(broadcast(sellers_df),'seller_id','inner')
full_orders_df=orders_items_products_seller_joined.join(customers_df,'customer_id','inner')
# Geolocation Data
full_orders_df=full_orders_df.join(broadcast(geolocation_df),full_orders_df.customer_zip_code_prefix==geolocation_df.geolocation_zip_code_prefix,'left')
full_orders_df=full_orders_df.join(broadcast(reviews_df),'order_id','left')
full_orders_df=full_orders_df.join(payments_df,'order_id','left')

In [ ]:
full_orders_df.cache()

# Aggregation

In [ ]:
# Total order per customer
total_order_per_customer=full_orders_df.groupBy('customer_id')\
                         .agg(count('order_id').alias('Total_orders'))\
                        .orderBy(col('Total_orders').desc())
total_order_per_customer.show(5)

In [ ]:
#Average Review per seller
seller_reviews = full_orders_df.groupBy('seller_id')\
    .agg(avg('review_score').alias('avg_review_score'))\
    .orderBy('avg_review_score',ascending=False)
seller_reviews.show(5)

In [ ]:
# Most Sold Products
top_products=full_orders_df.groupBy('product_id')\
.agg(count('order_id').alias('Most Sold Proucts'))\
.orderBy('Most Sold Proucts',ascending = False)
top_products.show(5)

In [ ]:
# Top 10 customer by spending
total_spend_per_customer=full_orders_df.groupBy('customer_id')\
.agg(sum('price').alias('TotalSpend'))\
.orderBy('TotalSpend',ascending=False)
total_spend_per_customer.show(10)

# Window Function And Ranking

In [ ]:
from pyspark.sql.window import Window

In [ ]:
window_spec=Window.partitionBy('seller_id').orderBy(desc('price'))
# Rank Top Selling Products Per Seller
rank_df=full_orders_df.withColumn('rank',rank().over(window_spec)).filter(col('rank')<=5)
rank_df.select('seller_id','price','rank').show(10)

# Advanced Aggregation

In [ ]:
# Total Revenue & Average Order per customer
customer_spending_df=full_orders_df.groupBy('customer_id')\
.agg(
            count('order_id').alias('Total_order'),
            sum('price').alias('Total_spend'),
            round(avg('price'),2).alias('AOV')
    )\
.orderBy(desc('Total_spend'))
customer_spending_df.show(10)

In [ ]:
# Seller performance Matrix(Revenue,Average Review,Order Count)
seller_performance=full_orders_df.groupBy('seller_id')\
.agg(
        count('order_id').alias('total_order'),
        sum('price').alias('Total_revenue'),
        round(avg('review_score'),2).alias('Average_review'),
        round(stddev('price'),2).alias('STD')
    )\
.orderBy(desc('Total_revenue')).show(10)

In [ ]:
# Monthly Revenue and order count
monthly_matrix_df=full_orders_df.dropDuplicates(['order_id', 'order_item_id']) \
    .withColumn('Year_Month',date_format('order_purchase_timestamp','yyyy-MM'))\
        .groupBy('Year_Month')\
.agg(
        countDistinct('order_id').alias('Total_Order'),
        sum('price').alias('Total_Revenue'),
        round(avg('price'),2).alias('Average_Order_Value'),
        max('price').alias('Max_Order_Value'),
        min('price').alias('Min_Order_Value')
    )\
.orderBy('Total_Revenue').show(10)

In [ ]:
# customer retention(first and last order)
customer_retention_df = full_orders_df.groupBy('customer_id') \
    .agg(
        min('order_purchase_timestamp').alias('First_Order_Date'),   # Finds oldest date
        max('order_purchase_timestamp').alias('Last_Order_Date'),    # Finds newest date
        count('order_id').alias('Total_Order_Made')          # Counts unique orders
    ) \
    .orderBy('Total_Order_Made',ascending=False).show(20)

# Extended Enrichment

In [ ]:
# Order Status Flags
order_status=full_orders_df.withColumn('is_delivered',when(col('order_status')=='delivered',lit(1)).otherwise(0))\
.withColumn('is_canceled',when(col('order_status')=='canceled',lit(1)).otherwise(0))
order_status.where(full_orders_df['order_status']!='delivered').select('order_status','is_delivered','is_canceled').show(20)

In [ ]:
# Order Revenue Calculation
full_orders_df=full_orders_df.withColumn('order_revenue',col('price')+col('freight_value'))
full_orders_df.select('price','freight_value','order_revenue').show(10)

In [ ]:
# Customer segmentation based on spending
customer_spending_df=customer_spending_df\
.withColumn('customer_segment'
            ,when(col('AOV')>=1200,'High-Value')
            .when((col('AOV')<1200)&(col('AOV')>=700),'Medium-Value')
            .otherwise('Low-Value')
           )
customer_spending_df.show(10)

In [ ]:
full_orders_df=full_orders_df.join(customer_spending_df.select('customer_id','customer_segment'),'customer_id',how='left')
full_orders_df.select('customer_segment').show(10)

In [ ]:
full_orders_df.select('customer_id','customer_segment').show(10)

In [ ]:
# Hourly Order Distribution
full_orders_df=full_orders_df.withColumn('hour_of_day',expr('hour(order_purchase_timestamp)'))
full_orders_df.select('order_purchase_timestamp','hour_of_day').show(10)

In [ ]:
# weekday vs weekend order
full_orders_df=full_orders_df.withColumn('order_day_type',when(expr('dayofweek(order_purchase_timestamp)IN(1,7)'),lit('weekend')).otherwise(lit('weekday')))

In [ ]:
full_orders_df.select('order_purchase_timestamp','order_day_type').show(10)

In [ ]:
 # Freight Cost analysis
full_orders_df=full_orders_df.withColumn('Freight_Category'
                                         ,when(col('freight_value')>=400,'High_cost')\
                                         .when((col('freight_value')<400) & (col('freight_value')>=200) , 'Medium_cost')\
                                         .otherwise('low_cost')
                                        )    


In [ ]:
full_orders_df.select('freight_value','Freight_Category').show(20)

In [ ]:
# Order Volume by customer state
full_orders_df=full_orders_df.groupBy('customer_state').agg(
                                sum('price').alias('Total_Revenue_Per_State')
                                                        )

In [ ]:
full_orders_df.select('customer_state','Total_Revenue_Per_State').show(28)